In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install xgboost

Import Libraries

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import VotingClassifier
import seaborn as sns
import matplotlib.pyplot as plt

Preprocessing and Combining Data

Import Health Indicators Dataset and Print Correlation Matrix/Feature Importance

In [ ]:
import xgboost as xgb
# Load dataset
health_indicators_dataset = pd.read_csv('/kaggle/input/diabetes-indicators-dataset/diabetes_012_health_indicators_BRFSS2021.csv')

health_indicators_dataset = health_indicators_dataset.apply(lambda x: x.str.strip() if x.dtype == "object" else x)

# Rename the outcome column to 'Diabetes_Status'

health_indicators_dataset.rename(columns={'Diabetes_012': 'Diabetes_Status'}, inplace=True)

# Rename the 'Sex' column to 'Gender
health_indicators_dataset.rename(columns={'Sex': 'Gender'}, inplace=True)

# Rename the 'PhysActivity" column to "PhysicallyActive"
health_indicators_dataset.rename(columns={'PhysActivity': 'PhysicallyActive'}, inplace=True)

# Rename the "Smoker" and "HvyHvyAlcoholConsump" to "Smoking" and "Alcohol" respectively
health_indicators_dataset.rename(columns={'Smoker': 'Smoking', 'HvyAlcoholConsump': 'Alcohol'}, inplace=True)


def find_median(a, b):
    return (a + b) / 2

# Mapping referenced from: https://www.cdc.gov/brfss/annual_data/2020/pdf/codebook20_llcp-v2-508.pdf

age_mapping = {
    1: find_median(18, 24),
    2: find_median(25, 29),
    3: find_median(30, 34),
    4: find_median(35, 39),
    5: find_median(40, 44),
    6: find_median(45, 49),
    7: find_median(50, 54),
    8: find_median(55, 59),
    9: find_median(60, 64),
    10: find_median(65, 69),
    11: find_median(70, 74),
    12: find_median(75, 79),
    13: 80
}

health_indicators_dataset['Age'] = health_indicators_dataset['Age'].map(age_mapping)

# Compute the correlation matrix
correlation_matrix = health_indicators_dataset.corr()

# Visualize the correlation matrix using a heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Correlation Matrix of Health Indicators Dataset')
plt.show()

from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split

# Split the data
X = health_indicators_dataset.drop('Diabetes_Status', axis=1)
y = health_indicators_dataset['Diabetes_Status']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train a simple model
model = XGBClassifier()
model.fit(X_train, y_train)

# Plot feature importance
xgb.plot_importance(model)
plt.show()



# printing the head of the dataset
print(health_indicators_dataset.head())

Import First Gestational Diabetes Dataset and Print Out Correlation Matrix/Feature Importance

In [ ]:
# load dataset
gestational_diabetes_dataset = pd.read_csv('/kaggle/input/gestational-diabetes-dataset/gestational_diabetes_dataset.csv')

gestational_diabetes_dataset = gestational_diabetes_dataset.apply(lambda x: x.str.strip() if x.dtype == "object" else x)

# Rename the outcome column to 'Diabetes_Status' and map values of 1 to 3
gestational_diabetes_dataset.rename(columns={'Outcome': 'Diabetes_Status'}, inplace=True)
# gestational_diabetes_dataset['Diabetes_Status'] = gestational_diabetes_dataset['Diabetes_Status'].map({0: 0, 1: 3})


# Define a function to classify BP levels based only on Diastolic BP
def classify_bp(row):
    dia_bp = row['BloodPressure']

    if dia_bp < 60:
        return 0  # Low BP
    elif 60 <= dia_bp <= 80:
        return 1  # Normal BP
    else:
        return 2  # High BP

# Apply the function to create the 'BPLevel' column based on diastolic pressure
gestational_diabetes_dataset['BPLevel'] = gestational_diabetes_dataset.apply(classify_bp, axis=1)

# Remove the 'Dia BP' column (and 'Sys BP' if no longer needed)
gestational_diabetes_dataset.drop(['BloodPressure'], axis=1, inplace=True)

# Compute the correlation matrix
correlation_matrix = gestational_diabetes_dataset.corr()

# Visualize the correlation matrix using a heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Correlation Matrix of Gestational Diabetes Dataset')
plt.show()


from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split

# Split the data
X = gestational_diabetes_dataset.drop('Diabetes_Status', axis=1)
y = gestational_diabetes_dataset['Diabetes_Status']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train a simple model
model = XGBClassifier()
model.fit(X_train, y_train)

# Plot feature importance
xgb.plot_importance(model)
plt.show()




# printing the head of the dataset
print(gestational_diabetes_dataset.head())

In [ ]:
gestational_diabetes_dataset['Diabetes_Status'] = gestational_diabetes_dataset['Diabetes_Status'].map({0: 0, 1: 3})

In [ ]:
missing_per_column = gestational_diabetes_dataset.isnull().sum()
print(missing_per_column)  # Displays the count of missing values for each column

Import BIT 2019 Dataset for more samples

In [ ]:
from sklearn.impute import KNNImputer
from sklearn.preprocessing import LabelEncoder
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
import xgboost as xgb

# import dataset
bit_2019_dataset = pd.read_csv('/kaggle/input/bit-2019-dataset/BIT_2019.csv')

bit_2019_dataset = bit_2019_dataset.apply(lambda x: x.str.strip() if x.dtype == "object" else x)

# rename column "highBP" to HighBP
bit_2019_dataset.rename(columns={'highBP': 'HighBP'}, inplace=True)

# rename column "Pregancies" to "Pregnancies"
bit_2019_dataset.rename(columns={'Pregancies': 'Pregnancies'}, inplace=True)

bit_2019_dataset['BPLevel'] = bit_2019_dataset['BPLevel'].str.lower()
# Strip any leading or trailing whitespace in the 'BPLevel' column
bit_2019_dataset['BPLevel'] = bit_2019_dataset['BPLevel'].str.strip()

# Function to classify diabetes status based on Diabetic, Pdiabetes, and Pregancies columns
def classify_diabetes(row):
    if row['Diabetic'] == 'no' and row['Pdiabetes'] == 0:
        return 'No Diabetes'
    elif row['Pdiabetes'] == 1:
        return 'Prediabetes'
    elif row['Diabetic'] == 'yes' and row['Pregnancies'] == 0:
        return 'Type-2 Diabetes'
    elif row['Diabetic'] == 'yes' and row['Pregnancies'] > 0 and row['Gender'] == 0: # only females
        return 'Gestational Diabetes'
    else:
        return 'Unknown'

# Fill missing values in 'Pdiabetes' and 'Diabetic' columns
bit_2019_dataset['Pdiabetes'].fillna('0', inplace=True)  # Assuming no prediabetes for missing values
bit_2019_dataset['Diabetic'].fillna('no', inplace=True)  # Assuming no diabetes for missing values

# Standardize the 'Pdiabetes' column to binary format (0 for no, 1 for yes)
bit_2019_dataset['Pdiabetes'] = bit_2019_dataset['Pdiabetes'].replace({'yes': 1, 'no': 0, '0': 0}).astype(int)

# Apply the classification function to create the target column 'Diabetes_Status'
bit_2019_dataset['Diabetes_Status'] = bit_2019_dataset.apply(classify_diabetes, axis=1)

# Check the distribution of the newly created 'Diabetes_Status' column
print(bit_2019_dataset['Diabetes_Status'].value_counts())

# Drop rows where 'Diabetes_Status' is 'Unknown'
bit_2019_dataset = bit_2019_dataset[bit_2019_dataset['Diabetes_Status'] != 'Unknown']

# Define a function to convert age ranges to numeric values (e.g., using the midpoint of the range)
def convert_age_range(age_range):
    if isinstance(age_range, str) and '-' in age_range:
        age_min, age_max = age_range.split('-')
        return (int(age_min) + int(age_max)) // 2
    else:
        return pd.to_numeric(age_range, errors='coerce')  # Handle any non-range values

# Apply the function to the Age column
bit_2019_dataset['Age'] = bit_2019_dataset['Age'].apply(convert_age_range)

# KNN Imputation for Age, Pregnancies, and BMI
imputer = KNNImputer(n_neighbors=5)
columns_to_impute = ['Age', 'Pregnancies', 'BMI']
bit_2019_dataset[columns_to_impute] = imputer.fit_transform(bit_2019_dataset[columns_to_impute])

# Identify all columns with 'object' dtype (categorical columns)
categorical_columns = bit_2019_dataset.select_dtypes(include=['object']).columns

# Initialize LabelEncoder
label_encoder = LabelEncoder()

# Apply label encoding to each categorical column
for column in categorical_columns:
    bit_2019_dataset[column] = label_encoder.fit_transform(bit_2019_dataset[column])
    # Print the mapping of original values to encoded values
    print(f"Mapping for column '{column}':")
    for i, item in enumerate(label_encoder.classes_):
        print(f"{item} --> {i}")
    print()  # Just to add a blank line between mappings

# Compute the correlation matrix
correlation_matrix = bit_2019_dataset.corr()
# Visualize the correlation matrix using a heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Correlation Matrix of BIT_2019 Dataset')
plt.show()

# Split the data
X = bit_2019_dataset.drop('Diabetes_Status', axis=1)
y = bit_2019_dataset['Diabetes_Status']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train a simple model
model = XGBClassifier()
model.fit(X_train, y_train)

# Plot feature importance
xgb.plot_importance(model)
plt.show()

# printing the head of the dataset
print(bit_2019_dataset.head())


Import Second Gestational Diabetes Dataset

In [ ]:
!pip install openpyxl

In [ ]:
# import dataset
gestational_diabetes_dataset2 = pd.read_excel('/kaggle/input/gestational-diabetes-dataset-2/gestational_diabetes_dataset2.xlsx')

gestational_diabetes_dataset2 = gestational_diabetes_dataset2.apply(lambda x: x.str.strip() if x.dtype == "object" else x)


# drop columns that are unnecessary for training
columns_to_drop = ["Case Number", "OGTT", "HDL", "Hemoglobin"]
gestational_diabetes_dataset2.drop(columns=columns_to_drop, inplace=True)

# Rename "No of Pregnancy" column to "Pregnancies", "Class Label(GDM /Non GDM)" to "Diabetes_Status", "Prediabetes" column to "Pdiabetes", "Family History" to "Family_Diabetes"
gestational_diabetes_dataset2.rename(columns={'No of Pregnancy': 'Pregnancies'}, inplace=True)
gestational_diabetes_dataset2.rename(columns={'Class Label(GDM /Non GDM)': 'Diabetes_Status'}, inplace=True)
# gestational_diabetes_dataset2['Diabetes_Status'] = gestational_diabetes_dataset2['Diabetes_Status'].map({0: 0, 1: 3})
gestational_diabetes_dataset2.rename(columns={'Prediabetes': 'Pdiabetes'}, inplace=True)
gestational_diabetes_dataset2.rename(columns={'Family History': 'Family_Diabetes'}, inplace=True)


def classify_bp(row):
    sys_bp = row['Sys BP']
    dia_bp = row['Dia BP']

    if sys_bp < 90 or dia_bp < 60:
        return 0  # Low BP
    elif 90 <= sys_bp <= 120 and 60 <= dia_bp <= 80:
        return 1  # Normal BP
    else:
        return 2  # High BP

# Apply the function to create the 'BPLevel' column
gestational_diabetes_dataset2['BPLevel'] = gestational_diabetes_dataset2.apply(classify_bp, axis=1)

# Remove the 'Sys BP' and 'Dia BP' columns
gestational_diabetes_dataset2.drop(['Sys BP', 'Dia BP'], axis=1, inplace=True)

# Compute the correlation matrix
correlation_matrix = gestational_diabetes_dataset2.corr()

# Visualize the correlation matrix using a heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Correlation Matrix of Gestational Diabetes 2 Dataset')
plt.show()


# Split the data
X = gestational_diabetes_dataset2.drop('Diabetes_Status', axis=1)
y = gestational_diabetes_dataset2['Diabetes_Status']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train a simple model
model = XGBClassifier()
model.fit(X_train, y_train)

# Plot feature importance
xgb.plot_importance(model)
plt.show()




print(gestational_diabetes_dataset2.head())

In [ ]:
gestational_diabetes_dataset2['Diabetes_Status'] = gestational_diabetes_dataset2['Diabetes_Status'].map({0: 0, 1: 3})

Add Another Prediabetes Dataset

In [ ]:
import pandas as pd

# Load dataset
prediabetes_dataset = pd.read_csv('/kaggle/input/prediabetes-dataset/prediabetes_dataset.csv')

# Strip any leading/trailing spaces in string columns
prediabetes_dataset = prediabetes_dataset.apply(lambda x: x.str.strip() if x.dtype == "object" else x)

# Rename 'class' to 'Diabetes_Status'
prediabetes_dataset.rename(columns={'class': 'Diabetes_Status'}, inplace=True)

# Map 'Diabetes_Status' and 'Gender' columns
prediabetes_dataset['Diabetes_Status'] = prediabetes_dataset['Diabetes_Status'].map({'Negative': 0, 'Positive': 1})
prediabetes_dataset['Gender'] = prediabetes_dataset['Gender'].map({'Female': 0, 'Male': 1})

# Create a dictionary for common mappings
binary_mapping = {'No': 0, 'Yes': 1}

# Define columns to apply the binary mapping
columns_to_map = ['Polyuria', 'Polydipsia', 'sudden weight loss', 'weakness', 'Polyphagia',
                  'Genital thrush', 'visual blurring', 'Itching', 'Irritability', 'delayed healing',
                  'partial paresis', 'muscle stiffness', 'Alopecia', 'Obesity']

# Apply the mapping to multiple columns
prediabetes_dataset[columns_to_map] = prediabetes_dataset[columns_to_map].replace(binary_mapping)

# print out correlaton map
correlation_matrix = prediabetes_dataset.corr()
plt.figure(figsize=(12, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Correlation Matrix of Prediabetes Dataset')
plt.show()

# Split the data
X = prediabetes_dataset.drop('Diabetes_Status', axis=1)
y = prediabetes_dataset['Diabetes_Status']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train a simple model
model = XGBClassifier()
model.fit(X_train, y_train)

# Plot feature importance
xgb.plot_importance(model)
plt.show()



# Output the updated dataset
print(prediabetes_dataset.head())


Check for Missing Values in Datasets

In [ ]:
# Function to print missing values in a dataset
def check_missing_values(dataset, dataset_name):
    missing_values = dataset.isnull().sum()  # Count missing values in each column
    missing_columns = missing_values[missing_values > 0]  # Filter columns with missing values

    if len(missing_columns) == 0:
        print(f"No missing values in {dataset_name}")
    else:
        print(f"{dataset_name} has missing values in the following columns:")
        print(missing_columns)


datasets = {'Health Indicators Dataset': health_indicators_dataset,
            'Prediabetes Dataset': prediabetes_dataset,
            'BIT 2019 Dataset': bit_2019_dataset,
            'Gestational Diabetes Dataset': gestational_diabetes_dataset,
            'Gestational Diabetes Dataset2': gestational_diabetes_dataset2}

# Check for missing values in each dataset
for name, dataset in datasets.items():
    check_missing_values(dataset, name)








Impute column 'BMI' in gestational_diabetes_dataset2

In [ ]:
imputer = KNNImputer(n_neighbors=5)
columns_to_impute = ['BMI']
gestational_diabetes_dataset2[columns_to_impute] = imputer.fit_transform(gestational_diabetes_dataset2[columns_to_impute])
check_missing_values(gestational_diabetes_dataset2, 'Gestational Diabetes Dataset2')

Correct Datatypes of each Dataset

In [ ]:
# Print out all the datatypes of each dataset
print("***** BEFORE CHANGE *****\n")
for name, dataset in datasets.items():
    print(f"===== {name} ======\n")
    print(dataset.dtypes)

health_indicators_dataset = health_indicators_dataset.astype({
    'Diabetes_Status': 'int64',
    'HighBP': 'category',
    'HighChol': 'category',
    'CholCheck': 'category',
    'BMI': 'float64',
    'Smoking': 'category',
    'Stroke': 'category',
    'HeartDiseaseorAttack': 'category',
    'PhysicallyActive': 'category',
    'Fruits': 'category',
    'Veggies': 'category',
    'Alcohol': 'category',
    'AnyHealthcare': 'category',
    'NoDocbcCost': 'category',
    'GenHlth': 'category',
    'MentHlth': 'int64',
    'PhysHlth': 'int64',
    'DiffWalk': 'category',
    'Gender': 'category',
    'Age': 'int64',
    'Education': 'category',
    'Income': 'category'
})
bit_2019_dataset = bit_2019_dataset.astype({
    'Age': 'int64',
    'Gender': 'category',
    'Family_Diabetes': 'category',
    'HighBP': 'category',
    'PhysicallyActive': 'category',
    'BMI': 'float64',
    'Smoking': 'category',
    'Alcohol': 'category',
    'Sleep': 'int64',
    'SoundSleep': 'int64',
    'RegularMedicine': 'category',
    'JunkFood': 'category',
    'Stress': 'category',
    'BPLevel': 'category',
    'Pregnancies': 'int64',
    'Pdiabetes': 'category',
    'UriationFreq': 'category',
    'Diabetic': 'category'
})
gestational_diabetes_dataset2 = gestational_diabetes_dataset2.astype({
    'Age': 'int64',
    'Pregnancies': 'int64',
    'Gestation in previous Pregnancy': 'category',
    'BMI': 'float64',
    'Family_Diabetes': 'category',
    'unexplained prenetal loss': 'category',
    'Large Child or Birth Default': 'category',
    'PCOS': 'category',
    'Sedentary Lifestyle': 'category',
    'Pdiabetes': 'category',
    'Diabetes_Status': 'category',
    'BPLevel': 'category'
})
prediabetes_dataset = prediabetes_dataset.astype({
    'Age': 'int64',
    'Gender': 'category',
    'Polyuria': 'category',
    'Polydipsia': 'category',
    'sudden weight loss': 'category',
    'weakness': 'category',
    'Polyphagia': 'category',
    'Genital thrush': 'category',
    'visual blurring': 'category',
    'Itching': 'category',
    'Irritability': 'category',
    'delayed healing': 'category',
    'partial paresis': 'category',
    'muscle stiffness': 'category',
    'Alopecia': 'category',
    'Obesity': 'category',
    'Diabetes_Status': 'category'
})

gestational_diabetes_dataset = gestational_diabetes_dataset.astype({
    'Pregnancies': 'int64',
    'Glucose': 'float64',
    'SkinThickness': 'float64',
    'Insulin': 'float64',
    'BMI': 'float64',
    'DiabetesPedigreeFunction': 'float64',
    'Age': 'int64',
    'Diabetes_Status': 'category',
    'BPLevel': 'category'
})






In [ ]:
print("***** AFTER CHANGE *****\n")
print("===== Health Indicators Dataset =====\n")
print(health_indicators_dataset.dtypes)
print("\n")
print("===== BIT 2019 Dataset =====\n")
print(bit_2019_dataset.dtypes)
print("\n")
print("===== Gestational Diabetes Dataset =====\n")
print(gestational_diabetes_dataset.dtypes)
print("\n")
print("===== Gestational Diabetes Dataset2 =====\n")
print(gestational_diabetes_dataset2.dtypes)
print("\n")
print("===== Prediabetes Dataset =====\n")
print(prediabetes_dataset.dtypes)
print("\n")


Separating Female and Male Datasets

In [ ]:
female_data = pd.concat([bit_2019_dataset[bit_2019_dataset['Gender'] == 0],
                         health_indicators_dataset[health_indicators_dataset['Gender'] == 0],
                         gestational_diabetes_dataset, gestational_diabetes_dataset2, prediabetes_dataset[prediabetes_dataset['Gender'] == 0]], axis=0)
male_data = pd.concat([bit_2019_dataset[bit_2019_dataset['Gender'] == 1],
                       health_indicators_dataset[health_indicators_dataset['Gender'] == 1], prediabetes_dataset[prediabetes_dataset['Gender'] == 1]], axis=0)

Save Female and Male Datasets to CSV files

In [ ]:
female_data.to_csv('female_data.csv', index=False)
male_data.to_csv('male_data.csv', index=False)

Print Datatypes of Male and Female Datasets

In [ ]:
print("==== Female Data ====")
print(female_data.dtypes)
print("\n")
print("==== Male Data ====")
print(male_data.dtypes)

Check Class Distribution of Both Female and Male Datasets

In [ ]:
# Check the distribution of Diabetes_Status in the male dataset to confirm no '3's (gestational diabetes)
print("==== Male Data ====")
print(male_data['Diabetes_Status'].value_counts())

# Check the distribution of Diabetes_Status in the female dataset
print("==== Female Data ====")
print(female_data['Diabetes_Status'].value_counts())

Impute Missing Data first before changing datatypes

In [ ]:
import pandas as pd
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import KNNImputer, IterativeImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Identify numerical and categorical columns
female_numerical_cols = female_data.select_dtypes(include=['float64', 'int64']).columns
female_categorical_cols = female_data.select_dtypes(include=['category']).columns

male_numerical_cols = male_data.select_dtypes(include=['float64', 'int64']).columns
male_categorical_cols = male_data.select_dtypes(include=['category']).columns

# KNN Imputer for numerical columns
knn_imputer = KNNImputer(n_neighbors=10)  # You can adjust n_neighbors as needed
female_data[female_numerical_cols] = knn_imputer.fit_transform(female_data[female_numerical_cols])

# Create a Pipeline for categorical imputation
categorical_imputer = Pipeline(steps=[
    ('imputer', IterativeImputer(estimator=DecisionTreeClassifier(), random_state=0))
])

# Impute categorical columns
female_data[female_categorical_cols] = categorical_imputer.fit_transform(female_data[female_categorical_cols])

male_data[male_numerical_cols] = knn_imputer.fit_transform(male_data[male_numerical_cols])
male_data[male_categorical_cols] = categorical_imputer.fit_transform(male_data[male_categorical_cols])





Change datatypes of Female and Male Data

In [ ]:
female_data = female_data.astype({
    'Age': 'int64',
    'Gender': 'category',
    'Family_Diabetes': 'category',
    'HighBP': 'category',
    'PhysicallyActive': 'category',
    'BMI': 'float64',
    'Smoking': 'category',
    'Alcohol': 'category',
    'Sleep': 'int64',
    'SoundSleep': 'int64',
    'RegularMedicine': 'category',
    'JunkFood': 'category',
    'Stress': 'category',
    'BPLevel': 'category',
    'Pregnancies': 'int64',
    'Pdiabetes': 'category',
    'UriationFreq': 'category',
    'Diabetic': 'category',
    'Diabetes_Status': 'category',
    'HighChol': 'category',
    'CholCheck': 'category',
    'Stroke': 'category',
    'HeartDiseaseorAttack': 'category',
    'Fruits': 'category',
    'Veggies': 'category',
    'AnyHealthcare': 'category',
    'NoDocbcCost': 'category',
    'GenHlth': 'category',
    'MentHlth': 'float64',
    'PhysHlth': 'float64',
    'DiffWalk': 'category',
    'Education': 'category',
    'Income': 'category',
    'Glucose': 'float64',
    'SkinThickness': 'float64',
    'Insulin': 'float64',
    'DiabetesPedigreeFunction': 'float64',
    'Gestation in previous Pregnancy': 'category',
    'unexplained prenetal loss': 'category',
    'Large Child or Birth Default': 'category',
    'PCOS': 'category',
    'Sedentary Lifestyle': 'category',
    'Polyuria': 'category',
    'Polydipsia': 'category',
    'sudden weight loss': 'category',
    'weakness': 'category',
    'Polyphagia': 'category',
    'Genital thrush': 'category',
    'visual blurring': 'category',
    'Itching': 'category',
    'Irritability': 'category',
    'delayed healing': 'category',
    'partial paresis': 'category',
    'muscle stiffness': 'category',
    'Alopecia': 'category',
    'Obesity': 'category'
})

male_data = male_data.astype({
    'Age': 'int64',
    'Gender': 'category',
    'Family_Diabetes': 'category',
    'HighBP': 'category',
    'PhysicallyActive': 'category',
    'BMI': 'float64',
    'Smoking': 'category',
    'Alcohol': 'category',
    'Sleep': 'int64',
    'SoundSleep': 'int64',
    'RegularMedicine': 'category',
    'JunkFood': 'category',
    'Stress': 'category',
    'BPLevel': 'category',
    'Pregnancies': 'int64',
    'Pdiabetes': 'category',
    'UriationFreq': 'category',
    'Diabetic': 'category',
    'Diabetes_Status': 'category',
    'HighChol': 'category',
    'CholCheck': 'category',
    'Stroke': 'category',
    'HeartDiseaseorAttack': 'category',
    'Fruits': 'category',
    'Veggies': 'category',
    'AnyHealthcare': 'category',
    'NoDocbcCost': 'category',
    'GenHlth': 'category',
    'MentHlth': 'float64',
    'PhysHlth': 'float64',
    'DiffWalk': 'category',
    'Education': 'category',
    'Income': 'category',
    'Polyuria': 'category',
    'Polydipsia': 'category',
    'sudden weight loss': 'category',
    'weakness': 'category',
    'Polyphagia': 'category',
    'Genital thrush': 'category',
    'visual blurring': 'category',
    'Itching': 'category',
    'Irritability': 'category',
    'delayed healing': 'category',
    'partial paresis': 'category',
    'muscle stiffness': 'category',
    'Alopecia': 'category',
    'Obesity': 'category'
})



Save final imputed female and male datasets

In [ ]:
# making sure there are no missing values still
# Check for missing values
missing_values = female_data.isnull().sum()
print(missing_values[missing_values > 0])  # Only show columns with missing values
print("")
missing_values = male_data.isnull().sum()
print(missing_values[missing_values > 0])  # Only show columns with missing values
print("")

female_data.to_csv('female_data.csv')
male_data.to_csv('male_data.csv')

Plotting Feature Importance for Female Dataset

In [ ]:
import pandas as pd
from imblearn.over_sampling import SMOTE
import matplotlib.pyplot as plt
import seaborn as sns

female_data = pd.read_csv('/kaggle/working/female_data.csv')

# Separate features (X) and target variable (y)
X = female_data.drop('Diabetes_Status', axis=1)
y = female_data['Diabetes_Status']

# Split data into training and testing sets
X_train_female, X_test_female, y_train_female, y_test_female = train_test_split(X, y, test_size=0.2, random_state=42)

# Apply SMOTE to the training data
smote = SMOTE(random_state=42)
X_train_smote_female, y_train_smote_female = smote.fit_resample(X_train_female, y_train_female)

# Initialize and train the XGBoost classifier
model = XGBClassifier(random_state=42)
model.fit(X_train_smote_female, y_train_smote_female)

# Get feature importances
feature_importances = model.feature_importances_

# Create a DataFrame for feature importances
importance_df = pd.DataFrame({'Feature': X.columns, 'Importance': feature_importances})
importance_df = importance_df.sort_values(by='Importance', ascending=False)

# Plot feature importances
plt.figure(figsize=(12, 12))
sns.barplot(x='Importance', y='Feature', data=importance_df)
plt.title('Feature Importance for Imputed Female Dataset (SMOTE)')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.show()

In [ ]:
plt.savefig("feature_importance_female_data.png", format="png", dpi=300, bbox_inches="tight")

Plot Feature Importance For Male Dataset

In [ ]:
male_data = pd.read_csv('/kaggle/working/male_data.csv')

# Separate features (X) and target variable (y)
X = male_data.drop('Diabetes_Status', axis=1)
y = male_data['Diabetes_Status']

# Split data into training and testing sets
X_train_male, X_test_male, y_train_male, y_test_male = train_test_split(X, y, test_size=0.2, random_state=42)

# Apply SMOTE to the training data
smote = SMOTE(random_state=42)
X_train_smote_male, y_train_smote_male = smote.fit_resample(X_train_male, y_train_male)

# Initialize and train the XGBoost classifier
model = XGBClassifier(random_state=42)
model.fit(X_train_smote_male, y_train_smote_male)

# Get feature importances
feature_importances = model.feature_importances_

# Create a DataFrame for feature importances
importance_df = pd.DataFrame({'Feature': X.columns, 'Importance': feature_importances})
importance_df = importance_df.sort_values(by='Importance', ascending=False)

# Plot feature importances
plt.figure(figsize=(12, 12))
sns.barplot(x='Importance', y='Feature', data=importance_df)
plt.title('Feature Importance for Imputed Male Dataset (SMOTE)')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.show()
plt.savefig("feature_importance_male_data.png", format="png", dpi=300, bbox_inches="tight")

Finding the Best Features to Train for Male Dataset using SHAP

In [ ]:
import shap
import matplotlib.pyplot as plt

# Fit model
model = xgb.XGBClassifier(random_state=42)
model.fit(X_train_smote_male, y_train_smote_male)

# Initialize SHAP Explainer and calculate SHAP values
explainer = shap.Explainer(model, X_train_smote_male)
shap_values = explainer(X_train_smote_male)

# Loop through each class and plot SHAP summary for each
for i in range(shap_values.values.shape[2]):
    plt.figure()  # Create a new figure for each plot
    shap.summary_plot(shap_values[..., i], X_train_smote_male, show=False)
    plt.title(f"Class {i} SHAP Summary")  # Set the title
    plt.savefig(f"shap_summary_class_male{i}.png")  # Save the plot as a PNG file
    plt.show()


Finding the Best Features to Train for Female Dataset using SHAP

In [ ]:
# Fit model
model = xgb.XGBClassifier(random_state=42)
model.fit(X_train_smote_female, y_train_smote_female)

# Initialize SHAP Explainer and calculate SHAP values
explainer = shap.Explainer(model, X_train_smote_female)
shap_values = explainer(X_train_smote_female)

# Loop through each class and plot SHAP summary for each
for i in range(shap_values.values.shape[2]):
    plt.figure()  # Create a new figure for each plot
    shap.summary_plot(shap_values[..., i], X_train_smote_female, show=False)
    plt.title(f"Class {i} SHAP Summary")  # Set the title
    plt.savefig(f"shap_summary_class_female{i}.png")  # Save the plot as a PNG file
    plt.show()
    
